# 거래 카테고리 분류기 — 학습 · 평가 · 아티팩트 생성

**금융 AI Challenge 2026 / 소비 악순환 예측 및 시나리오 제공 시스템**
담당: 정채영 (모델 ① 거래 카테고리 분류기)

---
## 이 노트북이 하는 일

```
소상공인시장진흥공단 상가(상권)정보 277만 건
         +
온라인·구독·배달 브랜드 사전 (직접 구축)
         ↓
  [5단계 캐스케이드 분류기 학습]
         ↓
artifacts/  merchant_dict.json · brand_dict.json · model.npz · category_map.csv
         ↓
  FastAPI 가 로드해서 사용자 거래내역을 분류
```

## 왜 단일 ML 모델이 아니라 캐스케이드인가

결론부터: **한 종류의 방법으로는 전체를 못 덮는다.**

| 문제 | 단일 ML로 풀리나 |
|---|---|
| 배달의민족·넷플릭스 분류 | ❌ 학습 데이터(상가정보)에 아예 없음 |
| "카드대금 이체" 처리 | ❌ 소비가 아닌데 억지로 카테고리를 찍음 |
| "토스페이먼츠" 처리 | ❌ 무엇을 샀는지 알 수 없는데 확신함 |
| "○○약국" 분류 | ⭕ 가능하지만 규칙이 더 정확하고 빠름 |
| "라라머시기" 같은 롱테일 | ⭕ **여기가 ML의 자리** |

그래서 **확실한 것부터 위에서 걷어내고, 남은 것만 아래로 내린다.**
위 단계일수록 정확하고, 아래 단계일수록 커버리지가 넓다.

In [11]:
import sys, glob, json, os, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd


# ---- 저장소 루트 찾기 ------------------------------------------------------
# 주피터를 어디서 띄우든(notebooks/, dajim-ml/, 저장소 루트) 동작해야 한다.
# 처음에는 Path.cwd().parent 로 고정했는데, VS Code 나 Jupyter Lab 이
# 작업 디렉터리를 다르게 잡으면 그대로 ModuleNotFoundError 가 났다.
# cwd 에서 위로 올라가며 패키지가 있는 위치를 직접 찾는 쪽이 안전하다.
def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "category_classifier" / "__init__.py").exists():
            return p
    raise RuntimeError(
        "category_classifier 패키지를 찾지 못했습니다.\n"
        f"현재 위치: {start}\n"
        "dajim-ml/notebooks/ 안에서 주피터를 실행했는지 확인하세요."
    )


ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

from category_classifier.taxonomy import (CATEGORIES, MID_MAP, SUB_OVERRIDE,
                                          code_to_category, korean, ALL_LABELS)
from category_classifier.brands import BRANDS, brand_stats
from category_classifier.classifier import (MerchantClassifier, CharNGramNB,
                                            normalize, build_merchant_dict,
                                            keyword_match)

ART = ROOT / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

print("ROOT         :", ROOT)
print("artifacts    :", ART)
print("카테고리      :", {k: korean(k) for k in ALL_LABELS})
print("브랜드 사전   :", brand_stats(), "| 합계", sum(brand_stats().values()))

ROOT         : /Users/curie/Documents/BEYOND 학회/금융AI/dajim-ml
artifacts    : /Users/curie/Documents/BEYOND 학회/금융AI/dajim-ml/artifacts
카테고리      : {'food': '식비', 'delivery': '배달', 'shopping': '쇼핑', 'transport': '교통', 'subscription': '구독', 'leisure': '여가', 'medical': '의료', 'education': '교육', 'living': '생활·고정', 'other': '기타'}
브랜드 사전   : {'delivery': 22, 'subscription': 75, 'shopping': 58, 'food': 71, 'transport': 39, 'leisure': 47, 'living': 29, 'medical': 3} | 합계 344


In [8]:
files = list(DATA_DIR.rglob("*.csv"))

print(f"총 CSV 파일: {len(files)}개\n")

for f in files:
    print("=" * 60)
    print(f"파일: {f}")

    for encoding in ["utf-8", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(
                f,
                nrows=5,
                encoding=encoding
            )

            print(f"인코딩: {encoding}")
            print(f"컬럼: {df.columns.tolist()}")
            print(df.head(2))
            break

        except UnicodeDecodeError:
            continue

    else:
        print("❌ 읽을 수 있는 인코딩을 찾지 못함")

총 CSV 파일: 17개

파일: /Users/curie/Documents/BEYOND 학회/금융AI/data/소상공인시장진흥공단_상가&#40;상권&#41;정보 업종코드_20230228.csv
인코딩: cp949
컬럼: ['대분류코드', '대분류명', '중분류코드', '중분류명', '소분류코드', '소분류명']
  대분류코드 대분류명 중분류코드              중분류명   소분류코드        소분류명
0    G2  소매업  G202  자동차 부품 및 내장품 소매업  G20201     타이어 소매업
1    G2  소매업  G202  자동차 부품 및 내장품 소매업  G20202  자동차 부품 소매업
파일: /Users/curie/Documents/BEYOND 학회/금융AI/data/소상공인시장진흥공단_상가&#40;상권&#41;정보_20260630/소상공인시장진흥공단_상가(상권)정보_대구_202606.csv
인코딩: utf-8
컬럼: ['상가업소번호', '상호명', '지점명', '상권업종대분류코드', '상권업종대분류명', '상권업종중분류코드', '상권업종중분류명', '상권업종소분류코드', '상권업종소분류명', '표준산업분류코드', '표준산업분류명', '시도코드', '시도명', '시군구코드', '시군구명', '행정동코드', '행정동명', '법정동코드', '법정동명', '지번코드', '대지구분코드', '대지구분명', '지번본번지', '지번부번지', '지번주소', '도로명코드', '도로명', '건물본번지', '건물부번지', '건물관리번호', '건물명', '도로명주소', '구우편번호', '신우편번호', '동정보', '층정보', '호정보', '경도', '위도']
                 상가업소번호       상호명  지점명 상권업종대분류코드 상권업종대분류명 상권업종중분류코드 상권업종중분류명 상권업종소분류코드  상권업종소분류명

---
## 1. 데이터 적재

17개 시도 파일을 합친다. 필요한 컬럼은 **상호명과 소분류코드 두 개뿐**이다.
주소·좌표까지 읽으면 메모리만 잡아먹고 분류에는 쓰이지 않는다.

In [13]:
# ===========================================================================
# 원본 데이터 위치
# ===========================================================================

MANUAL_DATA_DIR = ""

REQUIRED = [
    "상호명",
    "상권업종소분류코드",
    "상권업종소분류명"
]


def find_data_dir() -> Path:
    env = os.environ.get("DAJIM_SANGGA_DIR")

    candidates = [
        Path(x).expanduser()
        for x in (MANUAL_DATA_DIR, env)
        if x
    ] + [
        ROOT / "data",
        ROOT.parent / "data",
        ROOT.parent.parent / "data",
        Path.home() / "Downloads",
    ]

    tried = []

    for c in candidates:
        if not c.exists():
            tried.append(f"✗ {c} (폴더 없음)")
            continue

        # 여기서는 상위 data 폴더 자체를 반환
        # 하위 폴더까지 rglob으로 검색하기 위해서
        csv_files = list(c.rglob("*.csv"))

        if csv_files:
            return c

        tried.append(f"✗ {c} (csv 없음)")

    raise FileNotFoundError(
        "상가(상권)정보 CSV를 찾지 못했습니다.\n\n"
        + "\n".join(tried)
    )


# ===========================================================================
# CSV 자동 인코딩 읽기
# ===========================================================================

def read_csv_auto(path, **kwargs):
    for encoding in ["utf-8", "cp949", "euc-kr"]:
        try:
            df = pd.read_csv(
                path,
                encoding=encoding,
                **kwargs
            )

            return df, encoding

        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"읽을 수 있는 인코딩을 찾지 못했습니다: {path}"
    )


# ===========================================================================
# 데이터 위치 및 CSV 탐색
# ===========================================================================

DATA_DIR = find_data_dir()

files = sorted(DATA_DIR.rglob("*.csv"))

print(f"데이터 위치: {DATA_DIR}")
print(f"전체 CSV: {len(files)}개")


# ===========================================================================
# 실제 학습에 사용할 CSV 선별
# ===========================================================================
# 모든 CSV가 같은 구조가 아니므로
# REQUIRED 컬럼이 모두 존재하는 파일만 사용한다.

valid_files = []
ignored_files = []

for f in files:

    try:
        head, encoding = read_csv_auto(
            f,
            nrows=0
        )

        columns = set(head.columns)

        if all(col in columns for col in REQUIRED):
            valid_files.append((f, encoding))
        else:
            ignored_files.append((f, encoding))

    except Exception as e:
        print(f"⚠️ 읽기 실패: {f}")
        print(f"   {e}")


# ===========================================================================
# 파일 목록 확인
# ===========================================================================

print("\n" + "=" * 70)
print("✅ 학습 데이터로 사용할 파일")
print("=" * 70)

for f, encoding in valid_files:
    print(f"[{encoding}] {f}")


print("\n" + "=" * 70)
print("⏭️ 제외한 파일")
print("=" * 70)

for f, encoding in ignored_files:
    print(f"[{encoding}] {f}")


# ===========================================================================
# 데이터 적재
# ===========================================================================

if not valid_files:
    raise ValueError(
        "필요한 컬럼을 가진 CSV를 찾지 못했습니다."
    )


t0 = time.time()

dfs = []

for f, encoding in valid_files:

    temp = pd.read_csv(
        f,
        usecols=REQUIRED,
        encoding=encoding,
        low_memory=False
    )

    dfs.append(temp)

df = pd.concat(
    dfs,
    ignore_index=True
)

print(
    f"\n총 {len(df):,}건 적재 "
    f"({time.time() - t0:.1f}초)"
)

print("\n컬럼:")
print(df.columns.tolist())

print("\n샘플:")
display(df.head(3))

데이터 위치: /Users/curie/Documents/BEYOND 학회/금융AI/data
전체 CSV: 17개

✅ 학습 데이터로 사용할 파일
[utf-8] /Users/curie/Documents/BEYOND 학회/금융AI/data/소상공인시장진흥공단_상가&#40;상권&#41;정보_20260630/소상공인시장진흥공단_상가(상권)정보_강원_202606.csv
[utf-8] /Users/curie/Documents/BEYOND 학회/금융AI/data/소상공인시장진흥공단_상가&#40;상권&#41;정보_20260630/소상공인시장진흥공단_상가(상권)정보_경기_202606.csv
[utf-8] /Users/curie/Documents/BEYOND 학회/금융AI/data/소상공인시장진흥공단_상가&#40;상권&#41;정보_20260630/소상공인시장진흥공단_상가(상권)정보_경남_202606.csv
[utf-8] /Users/curie/Documents/BEYOND 학회/금융AI/data/소상공인시장진흥공단_상가&#40;상권&#41;정보_20260630/소상공인시장진흥공단_상가(상권)정보_경북_202606.csv
[utf-8] /Users/curie/Documents/BEYOND 학회/금융AI/data/소상공인시장진흥공단_상가&#40;상권&#41;정보_20260630/소상공인시장진흥공단_상가

,상호명,상권업종소분류코드,상권업종소분류명
0,파크랜드춘천,G21304,운동용품 소매업
1,한성자동차강릉서비스센터,S20302,자동차 세차장
2,제이킥시스템프로덕션,P10501,입시·교과학원


---
## 2. 업종코드 → 카테고리 매핑

### 왜 247개를 하나씩 쓰지 않았나

소분류 247개를 전부 손으로 매핑하면 두 가지가 나빠진다.
1. 오타·누락이 생겨도 발견이 어렵다
2. "왜 이 업종이 여기냐"를 247번 설명해야 한다

대신 **중분류 75개에 기본값을 주고, 상식과 어긋나는 소분류만 예외로 덮었다.**
예외는 12개뿐이다. 유지보수 대상이 247 → 87로 줄었고,
발표에서는 "중분류 단위로 매핑하고 12개만 예외 처리했습니다"로 한 문장에 끝난다.

예외의 예:
- `G213 오락용품 소매 = shopping` 인데 **자전거(G21305)는 transport** — 이동수단이므로
- `G215 의약·화장품 = shopping` 인데 **약국(G21501)은 medical**
- `G206 음료 소매 = food` 인데 **주류(G20602)는 leisure** — 주점과 같은 비필수 지출

In [14]:
df["category"] = df["상권업종소분류코드"].map(code_to_category)

dist = (df["category"].value_counts(normalize=True) * 100).round(2)
dist_df = pd.DataFrame({"비율(%)": dist, "건수": df["category"].value_counts()})
dist_df["한글"] = [korean(i) for i in dist_df.index]
print(dist_df.to_string())

           비율(%)      건수     한글
category                       
food       34.44  954735     식비
leisure    20.52  568952     여가
other      12.05  333956     기타
shopping   11.26  312076     쇼핑
living      9.12  252957  생활·고정
education   5.77  160057     교육
medical     3.60   99884     의료
transport   3.24   89867     교통


### ⚠️ 여기서 반드시 짚어야 할 사실

**delivery(배달)와 subscription(구독)이 0건이다.**

상가(상권)정보는 사업자등록된 **오프라인 점포**만 담는다.
배달의민족·넷플릭스처럼 점포가 없는 사업자는 데이터에 존재하지 않는다.

그런데 이 둘은 **우리 서비스가 행동 추천을 거는 핵심 카테고리**다.
(배달 줄이기, 구독 해지 — 시나리오 엔진의 1·2순위 행동)

즉 **ML을 아무리 잘 학습시켜도 이 두 카테고리는 구조적으로 0% 정확도다.**
브랜드 사전이 선택이 아니라 필수인 이유가 여기 있다.

In [15]:
# 카테고리별 대표 업종 확인 — 매핑이 상식과 맞는지 눈으로 검증
for cat in ["food", "shopping", "leisure", "transport", "living", "other"]:
    top = (df[df.category == cat]["상권업종소분류명"]
           .value_counts().head(6).index.tolist())
    print(f"{korean(cat):6s} ({cat:9s}) ← {', '.join(top)}")

식비     (food     ) ← 백반/한정식, 카페, 슈퍼마켓, 돼지고기 구이/찜, 편의점, 김밥/만두/분식
쇼핑     (shopping ) ← 기타 의류 소매업, 여성 의류 소매업, 화장품 소매업, 핸드폰 소매업, 꽃집, 그 외 기타 상품 전문 소매업
여가     (leisure  ) ← 미용실, 요리 주점, 펜션, 피부 관리실, 네일숍, 노래방
교통     (transport) ← 자동차 정비소, 자동차 세차장, 주유소, 자동차 부품 소매업, 자동차 대여업, 모터사이클 수리업
생활·고정  (living   ) ← 부동산 중개/대리업, 건축물 일반 청소업, 세탁소, 철물/공구 소매업, 가전제품 수리업, 그 외 기타 개인/가정용품 수리업
기타     (other    ) ← 경영 컨설팅업, 광고 대행업, 인테리어 디자인업, 건축 설계 및 관련 서비스업, 명함/간판/광고물 제작, 시각 디자인업


---
## 3. 정규화

같은 가맹점이 결제 표기마다 다르게 찍힌다.
```
'(주)스타벅스커피코리아'  '스타벅스 강남2호점'  'STARBUCKS'
```
이걸 하나로 모으지 않으면 사전이 무한히 커진다.

### 정규화 중 겪은 문제 — 지점명 제거

처음에는 `[가-힣]{1,6}점$` 정규식으로 '역삼점' 같은 지역 지점명을 지우려 했다.
그런데 정규식이 왼쪽부터 매칭하는 바람에 결과가 갈렸다.

```
'김밥천국역삼점'  → '밥천국역삼점'을 매칭 → 남는 글자 1자 → 스킵 (제거 실패)
'gs25서울대점'   → '서울대점'을 매칭    → 정상 제거
```

어느 글자가 지역명인지 규칙으로 알아낼 방법이 없다.
**그래서 규칙을 정교하게 만드는 대신 매칭 방식을 바꿨다.**

- 정규화는 `본점 / 2호점 / 1층`처럼 **명시적 표기만** 제거
- 지역 지점명은 사전 단계에서 **접두어 매칭**으로 흡수
  (`김밥천국역삼점` → 접두어 `김밥천국`이 사전에 있으면 매칭)

한국 상호는 브랜드가 앞, 지점이 뒤에 오므로 접두어 방향이 구조적으로 맞다.

In [16]:
demo = ["(주)스타벅스커피코리아 강남2호점", "김밥천국 역삼점", "GS25 서울대점",
        "㈜배달의민족", "NETFLIX.COM", "올리브영 홍대점 1층"]
pd.DataFrame({"원본": demo, "정규화": [normalize(x) for x in demo]})

,원본,정규화
0,(주)스타벅스커피코리아 강남2호점,스타벅스커피코리아강남
1,김밥천국 역삼점,김밥천국역삼점
2,GS25 서울대점,gs25서울대점
3,㈜배달의민족,배달의민족
4,NETFLIX.COM,netflixcom
5,올리브영 홍대점 1층,올리브영홍대점


In [17]:
t0 = time.time()
df["norm"] = df["상호명"].map(normalize)
df = df[df["norm"].str.len() >= 2].copy()
print(f"정규화 완료 ({time.time()-t0:.1f}초) | {len(df):,}건 / 고유 상호명 {df['norm'].nunique():,}개")

정규화 완료 (2.7초) | 2,763,850건 / 고유 상호명 1,834,676개


---
## 4. 학습/평가 분할 — 누수를 막는 방법

**행 단위로 랜덤 분할하면 안 된다.**

'스타벅스'는 데이터에 수백 번 나온다. 행 단위로 나누면 같은 이름이
train과 test 양쪽에 들어가고, 모델은 그냥 외운 걸 맞히면서 정확도가 부풀려진다.

그래서 **고유 상호명 단위로 분할**한다. 한 이름은 train이거나 test이지 둘 다일 수 없다.
이렇게 해야 "처음 보는 가맹점을 맞힐 수 있는가"라는 진짜 질문에 답하게 된다.

In [18]:
# 고유 상호명 → 최빈 카테고리 (groupby.apply 는 느려서 value_counts 방식 사용)
vc = df.groupby(["norm", "category"]).size().reset_index(name="n")
vc = vc.sort_values("n", ascending=False).drop_duplicates("norm")
uniq = vc[["norm", "category"]].reset_index(drop=True)

rng = np.random.default_rng(42)
mask = rng.random(len(uniq)) < 0.80
train_names = set(uniq.loc[mask, "norm"])
test_names = set(uniq.loc[~mask, "norm"])
print(f"train 고유명 {len(train_names):,} / test 고유명 {len(test_names):,}")

train_df = df[df["norm"].isin(train_names)]
test_df = df[df["norm"].isin(test_names)].drop_duplicates("norm")
print(f"train 행 {len(train_df):,} / test 행 {len(test_df):,}")

train 고유명 1,467,787 / test 고유명 366,889
train 행 2,213,097 / test 행 366,889


---
## 5. L3 — 상호명 사전 구축

상가정보에서 **"이 이름은 거의 항상 이 카테고리"**인 항목만 사전으로 승격한다.

두 가지 기준을 둔다.
- `min_count=3` : 3번 이상 등장해야 신뢰
- `min_purity=0.80` : 최빈 카테고리 비율이 80%를 넘어야 등재

**순도 기준이 중요하다.** '본점', '행복' 같은 이름은 여러 업종에 걸쳐 나타나는데,
이걸 사전에 넣으면 확신에 찬 오답을 만든다. 순도가 낮으면 사전에서 빼고 ML로 넘긴다.
**모르는 걸 모른다고 넘기는 게 틀린 답을 주는 것보다 낫다.**

In [19]:
t0 = time.time()
merchant_dict = build_merchant_dict(
    train_df["norm"].tolist(), train_df["category"].tolist(),
    min_count=3, min_purity=0.80)
print(f"사전 등재 {len(merchant_dict):,}개 ({time.time()-t0:.1f}초)")

md_cat = Counter(v[0] for v in merchant_dict.values())
print({korean(k): v for k, v in md_cat.most_common()})

# 순도 때문에 탈락한 이름 예시
# (groupby.apply 에 lambda 를 쓰면 147만 그룹에서 몇 분씩 걸린다. 벡터 연산으로 처리)
g = train_df.groupby(["norm", "category"]).size().reset_index(name="n")
g["total"] = g.groupby("norm")["n"].transform("sum")
best = g.sort_values("n", ascending=False).drop_duplicates("norm")
best["purity"] = best["n"] / best["total"]
rejected = best[(best["total"] >= 20) & (best["purity"] < 0.80)] \
    .sort_values("total", ascending=False)
print("\n[순도 미달로 사전에서 제외된 상호명 — 여러 업종에 걸쳐 있어 확신할 수 없음]")
print(rejected.head(10)[["norm", "total", "category", "purity"]].round(2).to_string(index=False))

사전 등재 67,244개 (2.2초)
{'식비': 26486, '여가': 15084, '생활·고정': 6420, '기타': 5792, '쇼핑': 5035, '의료': 3324, '교육': 2729, '교통': 2374}

[순도 미달로 사전에서 제외된 상호명 — 여러 업종에 걸쳐 있어 확신할 수 없음]
 norm  total category  purity
업소명없음   3571     food    0.23
  아지트    513  leisure    0.48
   다온    496     food    0.29
  큐사랑    399  leisure    0.60
신앙촌상회    322     food    0.72
   코코    315 shopping    0.66
  올리브    315 shopping    0.63
   소풍    293     food    0.57
  스카이    285  leisure    0.43
  에이스    279  leisure    0.48


---
## 6. L5 — ML 폴백 학습

### 왜 문자 n-gram인가 (형태소 분석기를 안 쓴 이유)

한국어 상호명은 **띄어쓰기가 거의 없거나 제멋대로**다.
`본죽&비빔밥카페`, `스타벅스강남점`, `쭈꾸미불백집`

형태소 분석기(KoNLPy 등)를 쓰면
- 신조어·브랜드명을 못 쪼갠다 (`쭈꾸미불백집` → 통째로 미등록어)
- JVM 의존성이 생겨 배포가 무거워진다 (무료 티어 메모리 512MB 환경에서 부담)

**문자 2~4그램은 띄어쓰기를 아예 안 본다.**
`약국`, `헤어`, `정비` 같은 업종 단서가 문자 단위로 잡히고, 의존성이 numpy 하나뿐이다.

### 왜 나이브베이즈인가

이 노트북은 **의존성 없이 바로 돌아가는 것**을 우선했다.
scikit-learn 의 `TfidfVectorizer + LinearSVC` 조합이 성능은 보통 더 좋지만,
팀원 환경마다 설치 상태가 다르면 재현이 안 된다.

나이브베이즈는 numpy만으로 구현 가능하고, 학습이 선형 1회 스캔이라 빠르다.
**sklearn 이 설치된 환경에서는 아래 셀에서 LinearSVC 로 교체할 수 있게 열어 뒀다.**

In [20]:
ml_train = uniq[uniq["norm"].isin(train_names)]
SAMPLE = 400_000                       # 롱테일 폴백이므로 전량 학습이 필요하지 않다
if len(ml_train) > SAMPLE:
    ml_train = ml_train.sample(SAMPLE, random_state=42)

t0 = time.time()
model = CharNGramNB(ngram_range=(2, 4), n_buckets=1 << 18, alpha=0.2, temperature=8.0)
model.fit(ml_train["norm"].tolist(), ml_train["category"].tolist())
print(f"학습 {len(ml_train):,}건 / {time.time()-t0:.1f}초 / 클래스 {model.classes_}")

학습 400,000건 / 3.5초 / 클래스 ['education', 'food', 'leisure', 'living', 'medical', 'other', 'shopping', 'transport']


### 신뢰도 보정 — 처음엔 전부 0.9999가 나왔다

나이브베이즈는 n-gram을 독립으로 가정해 확률을 전부 곱한다.
그래서 글자가 길수록 로그점수 차이가 벌어지고, softmax가 항상 1.0에 붙어버린다.
**신뢰도가 전부 1.0이면 임계값으로 '모름'을 걸러낼 수가 없다.**

n-gram 개수로 나눠 '평균 로그확률'로 바꾸니 길이에 무관한 신뢰도가 나왔다.

In [21]:
probe = ["행복한동네정육", "별빛헤어살롱", "현대오토서비스", "한빛수학전문",
         "zzqx", "김철수", "ㅁㄴㅇㄹ"]
pd.DataFrame([{"입력": t, "예측": korean(model.predict_one(t)[0]),
               "신뢰도": round(model.predict_one(t)[1], 3)} for t in probe])

,입력,예측,신뢰도
0,행복한동네정육,식비,0.978
1,별빛헤어살롱,여가,1.000
2,현대오토서비스,교통,1.000
3,한빛수학전문,교육,1.000
4,zzqx,교육,0.460
5,김철수,여가,0.614
6,ㅁㄴㅇㄹ,의료,0.664


---
## 7. 캐스케이드 조립 & 평가

In [22]:
clf = MerchantClassifier(merchant_dict=merchant_dict, model=model, ml_threshold=0.45)

t0 = time.time()
preds = clf.predict_many(test_df["상호명"].tolist())
print(f"추론 {len(preds):,}건 / {time.time()-t0:.1f}초 "
      f"({len(preds)/(time.time()-t0):,.0f}건/초)")

res = pd.DataFrame(preds)
res["true"] = test_df["category"].values
res["correct"] = res["category"] == res["true"]

추론 366,889건 / 5.1초 (72,323건/초)


In [23]:
# ---- 단계별 커버리지 & 정확도 ----
layer = res.groupby("layer").agg(
    건수=("correct", "size"),
    정확도=("correct", "mean"),
).reset_index()
layer["커버리지(%)"] = (layer["건수"] / len(res) * 100).round(2)
layer["정확도"] = layer["정확도"].round(4)
layer = layer.sort_values("건수", ascending=False)
print(layer.to_string(index=False))

known = res[res["category"] != "unknown"]
print(f"\n전체 정확도(미분류 포함) : {res['correct'].mean():.4f}")
print(f"분류된 건만의 정확도      : {known['correct'].mean():.4f}")
print(f"미분류율                 : {(res['category']=='unknown').mean():.4f}")

          layer     건수    정확도  커버리지(%)
          L5_ml 198957 0.6527    54.23
     L4_keyword 117198 0.8470    31.94
    L3_merchant  31454 0.9539     8.57
  L5_ml_lowconf  10537 0.0000     2.87
       L2_brand   8738 0.9122     2.38
          L1_pg      3 0.0000     0.00
L1_non_spending      2 0.0000     0.00

전체 정확도(미분류 포함) : 0.7280
분류된 건만의 정확도      : 0.7496
미분류율                 : 0.0287


**읽는 법:** 위 단계일수록 정확도가 높고 커버리지가 좁아야 정상이다.
만약 L5(ML)의 정확도가 L4(키워드)보다 높다면 키워드 규칙에 오류가 있다는 뜻이다.

In [24]:
# ---- 카테고리별 Precision / Recall / F1 ----
def prf(res):
    rows = []
    for c in sorted(set(res["true"]) | set(res["category"])):
        if c == "unknown":
            continue
        tp = ((res["category"] == c) & (res["true"] == c)).sum()
        fp = ((res["category"] == c) & (res["true"] != c)).sum()
        fn = ((res["true"] == c) & (res["category"] != c)).sum()
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        rows.append({"카테고리": korean(c), "code": c, "support": int(tp + fn),
                     "precision": round(p, 3), "recall": round(r, 3), "f1": round(f, 3)})
    return pd.DataFrame(rows).sort_values("support", ascending=False)

prf_df = prf(res)
print(prf_df.to_string(index=False))

# support 가 0인 클래스를 macro 평균에 넣으면 안 된다.
# delivery·subscription 은 '모델이 못 맞힌 것'이 아니라 '평가할 데이터가 없는 것'이다.
# 이 둘을 0점으로 넣으면 macro-F1 이 0.53 으로 떨어져 실제보다 나쁘게 보인다.
evaluable = prf_df[prf_df["support"] > 0]
print(f"\n평가 가능한 클래스 : {len(evaluable)}개 "
      f"(delivery·subscription·transfer 는 상가정보에 표본이 없어 제외)")
print(f"macro-F1    : {evaluable['f1'].mean():.4f}")
print(f"weighted-F1 : {(evaluable['f1']*evaluable['support']).sum()/evaluable['support'].sum():.4f}")

    카테고리         code  support  precision  recall    f1
      식비         food   123563      0.853   0.815 0.833
      여가      leisure    70677      0.670   0.747 0.706
      기타        other    50389      0.729   0.658 0.692
      쇼핑     shopping    39485      0.524   0.548 0.536
   생활·고정       living    30343      0.857   0.618 0.718
      교육    education    26381      0.784   0.714 0.747
      의료      medical    13051      0.839   0.876 0.857
      교통    transport    13000      0.770   0.751 0.761
      배달     delivery        0      0.000   0.000 0.000
      구독 subscription        0      0.000   0.000 0.000
transfer     transfer        0      0.000   0.000 0.000

평가 가능한 클래스 : 8개 (delivery·subscription·transfer 는 상가정보에 표본이 없어 제외)
macro-F1    : 0.7312
weighted-F1 : 0.7398


> **support가 0인 delivery·subscription은 여기서 평가할 수 없다.**
> 상가정보에 존재하지 않는 카테고리이기 때문이다.
> 이 둘은 아래 §8의 실전 표기 테스트로 따로 검증한다.

In [25]:
# ---- 혼동 행렬 (상위 오답 쌍) ----
conf = (res[~res["correct"] & (res["category"] != "unknown")]
        .groupby(["true", "category"]).size()
        .reset_index(name="건수").sort_values("건수", ascending=False).head(12))
conf["실제"] = conf["true"].map(korean)
conf["예측"] = conf["category"].map(korean)
print(conf[["실제", "예측", "건수"]].to_string(index=False))

   실제 예측   건수
   식비 여가 9661
   여가 식비 6690
   식비 쇼핑 5987
   쇼핑 여가 5950
   쇼핑 식비 4679
   기타 여가 4642
   기타 쇼핑 4263
   여가 쇼핑 4193
   쇼핑 기타 3019
생활·고정 기타 2944
생활·고정 쇼핑 2904
   기타 식비 2868


### 이 정확도를 그대로 서비스 정확도로 읽으면 안 된다

테스트셋은 **train 에 한 번도 안 나온 가맹점 이름만** 모아 놓은 집합이다.
즉 '동네 개인 가게 36만 곳'이고, 사전이 절대 못 맞히는 최악 조건이다.

실제 사용자 카드 내역은 다르다. 스타벅스·GS25·배달의민족처럼
**소수의 가맹점이 거래 대부분을 차지**한다. 아래는 그 차이를 실측한 것이다.

In [26]:
row_cov = df["norm"].isin(merchant_dict).mean()
uniq_cov = uniq["norm"].isin(merchant_dict).mean()
print(f"고유 상호명 기준 사전 커버리지 : {uniq_cov:6.2%}   ← 위 평가가 보는 세계")
print(f"실제 거래(행) 기준 커버리지    : {row_cov:6.2%}   ← 서비스가 보는 세계")
print()
print(f"→ 같은 사전인데 거래 기준으로 보면 커버리지가 {row_cov/uniq_cov:.1f}배로 커진다.")
print("  실제 카드내역은 체인점 비중이 상가정보보다 훨씬 높으므로")
print("  L3(정확도 95%) 비중은 이보다 더 커질 것으로 예상되지만,")
print("  ⚠️ 이건 아직 추정이다. 실제 카드내역 샘플로 검증해야 한다.")

고유 상호명 기준 사전 커버리지 :  3.67%   ← 위 평가가 보는 세계
실제 거래(행) 기준 커버리지    : 18.62%   ← 서비스가 보는 세계

→ 같은 사전인데 거래 기준으로 보면 커버리지가 5.1배로 커진다.
  실제 카드내역은 체인점 비중이 상가정보보다 훨씬 높으므로
  L3(정확도 95%) 비중은 이보다 더 커질 것으로 예상되지만,
  ⚠️ 이건 아직 추정이다. 실제 카드내역 샘플로 검증해야 한다.


---
## 8. 실전 결제 표기 테스트 ⭐

**상가정보로만 평가하면 서비스 정확도를 과대평가한다.**

실제 카드 내역에는 상가정보에 없는 것들이 잔뜩 섞여 있다.
온라인 브랜드, 결제대행사, 이체, 카드대금. 이걸 못 다루면 서비스가 망가진다.

그래서 **실제 결제 표기 형태로 직접 만든 테스트 셋**을 따로 둔다.
이 셋은 팀원 누구나 자기 카드 내역을 보고 계속 추가할 수 있다.

In [28]:
REAL_WORLD_CASES = [
    # (결제 표기, 정답 카테고리)
    ("배달의민족", "delivery"), ("(주)우아한형제들", "delivery"),
    ("쿠팡이츠", "delivery"), ("요기요", "delivery"),
    ("NETFLIX.COM", "subscription"), ("유튜브프리미엄", "subscription"),
    ("스포티파이", "subscription"), ("(주)멜론", "subscription"),
    ("OPENAI *CHATGPT SUBSCR", "subscription"), ("NOTION LABS", "subscription"),
    ("쿠팡(주)", "shopping"), ("무신사", "shopping"), ("올리브영 홍대점", "shopping"),
    ("11번가", "shopping"), ("다이소 강남점", "shopping"),
    ("스타벅스커피 코리아", "food"), ("GS25 신촌점", "food"), ("CU 역삼점", "food"),
    ("(주)파리크라상 파리바게뜨", "food"), ("맥도날드 강남", "food"),
    ("마켓컬리", "food"), ("이마트24 서울대", "food"),
    ("카카오T 택시", "transport"), ("쏘카", "transport"), ("SK에너지 주유소", "transport"),
    ("코레일 승차권", "transport"), ("티머니 충전", "transport"),
    ("CGV 강남", "leisure"), ("야놀자", "leisure"), ("에어비앤비", "leisure"),
    ("STEAMGAMES.COM", "leisure"), ("메가박스 코엑스", "leisure"),
    ("SKT 통신요금", "living"), ("한국전력공사", "living"), ("삼성화재 보험료", "living"),
    ("행복한약국", "medical"), ("서울대치과의원", "medical"),
    ("한빛수학학원", "education"), ("YBM어학원", "education"),
    # 소비가 아닌 것 — 반드시 걸러야 한다
    ("카드대금 결제", "transfer"), ("계좌이체", "transfer"),
    ("ATM 현금출금", "transfer"), ("리볼빙 약정", "transfer"),
    # 판단 불가 — 억지로 찍으면 안 된다
    ("토스페이먼츠(주)", "unknown"), ("NICEPAY", "unknown"), ("(주)KCP", "unknown"),
]

rw = pd.DataFrame(REAL_WORLD_CASES, columns=["표기", "정답"])
rw_pred = [clf.predict(x) for x in rw["표기"]]
rw["예측"] = [p["category"] for p in rw_pred]
rw["단계"] = [p["layer"] for p in rw_pred]
rw["근거"] = [p["evidence"] for p in rw_pred]
rw["정답여부"] = rw["예측"] == rw["정답"]

print(f"실전 표기 정확도: {rw['정답여부'].mean():.3f}  ({rw['정답여부'].sum()}/{len(rw)})")
print()
print(rw[~rw["정답여부"]].to_string(index=False) if (~rw["정답여부"]).any() else "전부 정답")

실전 표기 정확도: 1.000  (46/46)

전부 정답


In [29]:
rw.groupby("단계").agg(건수=("정답여부", "size"), 정확도=("정답여부", "mean")).round(3)

,건수,정확도
단계,,
L1_non_spending,4,1.0
L1_pg,3,1.0
L2_brand,32,1.0
L3_merchant,3,1.0
L4_keyword,4,1.0


### 개발 중 잡은 버그 — 부분 문자열 매칭의 함정

처음에는 이체 판별을 `["이체","대출","주식","세금",...] 부분 문자열 포함` 으로 짰다.
상가정보 277만 건에 돌려 보니 **멀쩡한 가맹점 133곳이 "소비 아님"으로 지워졌다.**

```
성주식당            → '주식'   에 걸림
한국세탁소           → '국세'   에 걸림
현대출력센타          → '대출'   에 걸림
커피빈여의도교보증권점    → '증권'   에 걸림
```

**카테고리를 틀리는 것보다 나쁘다.** 지출 자체가 통계에서 사라지기 때문이다.
그래서 두 가지로 바꿨다.

1. **CSV 의 `거래유형` 컬럼을 1순위로** 사용 (가맹점명 추측은 폴백일 뿐)
2. 가맹점명 판별은 **정확 일치 + 앵커된 패턴**만 허용 (`^카드대금`, `이체$`)

실제 카드사 CSV에는 거래유형 컬럼이 거의 항상 있다.
있는 정보를 안 쓰고 이름에서 추측하려 한 게 애초에 설계 실수였다.

In [30]:
# 오탐 재발 방지 테스트 — 위 4건이 정상 분류되는지 확인
regression = ["성주식당", "한국세탁소", "현대출력센타", "커피빈여의도교보증권점",
              "카드대금 결제", "계좌이체", "ATM출금"]
pd.DataFrame([{"표기": t, "카테고리": clf.predict(t)["category"],
               "단계": clf.predict(t)["layer"]} for t in regression])

,표기,카테고리,단계
0,성주식당,food,L3_merchant
1,한국세탁소,living,L3_merchant
2,현대출력센타,other,L5_ml
3,커피빈여의도교보증권점,food,L2_brand
4,카드대금 결제,transfer,L1_non_spending
5,계좌이체,transfer,L1_non_spending
6,ATM출금,transfer,L1_non_spending


---
## 9. 임계값 튜닝 — 모르는 걸 모른다고 하기

ML 신뢰도 임계값을 올리면 정확도는 오르지만 미분류가 늘어난다.

**우리 서비스에서는 미분류가 오답보다 낫다.**
카테고리를 잘못 찍으면 그 위의 소비 예측·ETA·행동 추천이 전부 조용히 틀어진다.
미분류는 최소한 사용자에게 "이건 뭐였나요?"라고 물어볼 수 있다.

In [31]:
sweep = []
for th in [0.0, 0.2, 0.3, 0.4, 0.45, 0.5, 0.6, 0.7, 0.8]:
    c2 = MerchantClassifier(merchant_dict=merchant_dict, model=model, ml_threshold=th)
    sample = test_df.sample(min(30_000, len(test_df)), random_state=1)
    pr = pd.DataFrame(c2.predict_many(sample["상호명"].tolist()))
    pr["true"] = sample["category"].values
    unk = (pr["category"] == "unknown").mean()
    ok = pr[pr["category"] != "unknown"]
    sweep.append({"임계값": th, "미분류율": round(unk, 4),
                  "분류건 정확도": round((ok["category"] == ok["true"]).mean(), 4),
                  "전체 정확도": round((pr["category"] == pr["true"]).mean(), 4)})
sweep_df = pd.DataFrame(sweep)
print(sweep_df.to_string(index=False))

 임계값   미분류율  분류건 정확도  전체 정확도
0.00 0.0000   0.7345  0.7345
0.20 0.0000   0.7345  0.7345
0.30 0.0013   0.7351  0.7342
0.40 0.0142   0.7417  0.7312
0.45 0.0291   0.7489  0.7271
0.50 0.0485   0.7579  0.7211
0.60 0.0985   0.7799  0.7031
0.70 0.1461   0.8013  0.6842
0.80 0.1968   0.8247  0.6624


---
## 10. 아티팩트 저장

여기서 나온 파일들이 **FastAPI 서버에 올라가는 전부**다.
원본 277만 건은 서버에 올리지 않는다.

In [32]:
clf.save(ART)

# 업종코드 → 카테고리 매핑표 (팀 공용 문서 겸 산출물)
cmap = (df.drop_duplicates("상권업종소분류코드")[["상권업종소분류코드", "상권업종소분류명", "category"]]
        .rename(columns={"상권업종소분류코드": "소분류코드", "상권업종소분류명": "소분류명"}))
cmap["중분류코드"] = cmap["소분류코드"].str[:4]
cmap["카테고리한글"] = cmap["category"].map(korean)
cmap["예외처리"] = cmap["소분류코드"].isin(SUB_OVERRIDE).map({True: "O", False: ""})
cmap = cmap.sort_values("소분류코드")[["소분류코드", "소분류명", "중분류코드",
                                    "category", "카테고리한글", "예외처리"]]
cmap.to_csv(ART / "category_map.csv", index=False, encoding="utf-8-sig")

# 평가 리포트
report = {
    "generated_at": time.strftime("%Y-%m-%d %H:%M"),
    "source_rows": int(len(df)),
    "unique_merchants": int(df["norm"].nunique()),
    "merchant_dict_size": len(merchant_dict),
    "brand_dict_size": sum(brand_stats().values()),
    "test_accuracy_all": round(float(res["correct"].mean()), 4),
    "test_accuracy_classified": round(float(known["correct"].mean()), 4),
    "unknown_rate": round(float((res["category"] == "unknown").mean()), 4),
    "macro_f1_evaluable": round(float(evaluable["f1"].mean()), 4),
    "real_world_accuracy": round(float(rw["정답여부"].mean()), 4),
    "layer_coverage": layer.set_index("layer")["커버리지(%)"].to_dict(),
}
json.dump(report, open(ART / "eval_report.json", "w"), ensure_ascii=False, indent=2)

for p in sorted(ART.iterdir()):
    print(f"{p.name:24s} {p.stat().st_size/1024:9.1f} KB")
print()
print(json.dumps(report, ensure_ascii=False, indent=2))

# 표본이 너무 작으면 배포용으로 쓰면 안 된다는 걸 눈에 띄게 알린다.
if report["source_rows"] < 1_000_000:
    print("\n⚠️  학습 표본이 {:,}건입니다. 전체 원본(약 276만건)이 아니므로".format(report["source_rows"]))
    print("    이 artifacts 를 서비스에 그대로 쓰면 안 됩니다. 커밋도 하지 마세요.")

brand_dict.json                7.0 KB
category_map.csv              12.3 KB
eval_report.json               0.5 KB
merchant_dict.json          2438.3 KB
model.npz                   1221.6 KB

{
  "generated_at": "2026-08-19 20:16",
  "source_rows": 2763850,
  "unique_merchants": 1834676,
  "merchant_dict_size": 67244,
  "brand_dict_size": 344,
  "test_accuracy_all": 0.728,
  "test_accuracy_classified": 0.7496,
  "unknown_rate": 0.0287,
  "macro_f1_evaluable": 0.7312,
  "real_world_accuracy": 1.0,
  "layer_coverage": {
    "L5_ml": 54.23,
    "L4_keyword": 31.94,
    "L3_merchant": 8.57,
    "L5_ml_lowconf": 2.87,
    "L2_brand": 2.38,
    "L1_pg": 0.0,
    "L1_non_spending": 0.0
  }
}


---
## 11. FastAPI 연동 예시

서버는 아티팩트만 로드하면 된다. 원본 데이터도, 학습 코드도 필요 없다.

```python
from fastapi import FastAPI, UploadFile
from classifier import MerchantClassifier
import pandas as pd, io

app = FastAPI()
CLF = MerchantClassifier.load("artifacts")     # 서버 기동 시 1회

@app.post("/api/v1/classify")
async def classify(file: UploadFile):
    tx = pd.read_csv(io.BytesIO(await file.read()))
    out = [CLF.predict(n) for n in tx["가맹점명"]]
    tx["category"]   = [o["category"]   for o in out]
    tx["confidence"] = [o["confidence"] for o in out]

    spend = tx[~tx.category.isin(["transfer", "unknown"])]
    return {
        "by_category": spend.groupby("category")["amount"].sum().abs().to_dict(),
        "unknown": tx[tx.category == "unknown"][["가맹점명", "amount"]].to_dict("records"),
        "excluded_transfer": int((tx.category == "transfer").sum()),
    }
```

`unknown` 을 응답에 담아 보내는 게 중요하다.
프론트에서 **"이 거래는 어떤 항목인가요?"** 로 사용자에게 물어보고,
그 답을 사전에 추가하면 쓸수록 정확해지는 구조가 된다.

In [ ]:
# 실제 CSV 처리 시뮬레이션
sample_tx = pd.DataFrame({
    "date": ["2026-08-01", "2026-08-02", "2026-08-02", "2026-08-03",
             "2026-08-04", "2026-08-05", "2026-08-05", "2026-08-06"],
    "가맹점명": ["배달의민족", "스타벅스 강남점", "NETFLIX.COM", "김철수",
                "GS25 역삼점", "무신사", "토스페이먼츠(주)", "카카오T 택시"],
    "거래유형": ["승인", "승인", "승인", "이체", "승인", "승인", "승인", "승인"],
    "amount": [-23000, -5600, -13500, -450000, -8200, -89000, -34000, -12400],
})
# 거래유형 컬럼을 함께 넘긴다 — '김철수'는 이름만으로는 절대 알 수 없다
out = [clf.predict(n, t) for n, t in zip(sample_tx["가맹점명"], sample_tx["거래유형"])]
sample_tx["category"] = [korean(o["category"]) if o["category"] in CATEGORIES
                         else o["category"] for o in out]
sample_tx["layer"] = [o["layer"] for o in out]
print(sample_tx.to_string(index=False))

spend = [o for o in out if o["category"] not in ("transfer", "unknown")]
tot = sum(-a for a, o in zip(sample_tx["amount"], out)
          if o["category"] not in ("transfer", "unknown"))
print(f"\n실제 소비 합계 : {tot:,}원  (이체·미분류 제외)")
print(f"단순 합산했다면: {-sample_tx['amount'].sum():,}원  ← 이체 45만원이 섞여 4배 부풀어난다")

> **마지막 두 줄이 이 모델의 존재 이유다.**
> 이체를 걸러내지 않으면 월 소비가 3배로 부풀고,
> 그 위에 올라가는 ETA·시나리오·행동 추천이 전부 무의미해진다.

---
## 다음 할 일

| 항목 | 비고 |
|---|---|
| 브랜드 사전 확장 (본인 카드내역 기반) | `brands.py`에 한 줄씩 추가 |
| `REAL_WORLD_CASES` 확충 | 실제 표기 100건 목표 |
| 순도 미달 상호명 수기 검토 | §5 rejected 목록 |
| sklearn LinearSVC 비교 실험 | NB 대비 F1 개선폭 확인 |
| AI Hub 업종 → 본 카테고리 매핑 | 팀원 모델 입력 생성용 |
| `/api/v1/classify` 연동 | §11 스키마 |